In [ ]:
import random
from collections import deque
import torch


class ReplayBuffer:
    def __init__(self, capacity=10000, batch_size=32, device="cpu"):
        self.memory = deque(maxlen=capacity)
        self.batch_size = batch_size
        self.device = device

    # push — store new experience
    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    # sample — get random batch for training
    def sample(self):
        batch = random.sample(self.memory, self.batch_size)

        states, actions, rewards, next_states, dones = zip(*batch)

        states      = torch.tensor(states, dtype=torch.float32).to(self.device)
        actions     = torch.tensor(actions, dtype=torch.long).to(self.device)
        rewards     = torch.tensor(rewards, dtype=torch.float32).to(self.device)
        next_states = torch.tensor(next_states, dtype=torch.float32).to(self.device)
        dones       = torch.tensor(dones, dtype=torch.float32).to(self.device)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.memory)

    def is_ready(self):
        return len(self.memory) >= self.batch_size

In [ ]:
import numpy as np

class PrioritizedReplayBuffer:
    def __init__(self, capacity=10000, batch_size=32, alpha=0.6, device="cpu"):
        self.capacity = capacity
        self.batch_size = batch_size
        self.alpha = alpha
        self.device = device

        self.memory = []
        self.priorities = np.zeros(capacity, dtype=np.float32)

        self.pos = 0
        self.size = 0


    def push(self, state, action, reward, next_state, done):
        max_priority = self.priorities.max() if self.size > 0 else 1.0

        if self.size < self.capacity:
            self.memory.append((state, action, reward, next_state, done))
        else:
            self.memory[self.pos] = (state, action, reward, next_state, done)

        self.priorities[self.pos] = max_priority

        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)


    def sample(self):
        probs = self.priorities[:self.size] ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(self.size, self.batch_size, p=probs)

        batch = [self.memory[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)

        states      = torch.tensor(states, dtype=torch.float32).to(self.device)
        actions     = torch.tensor(actions, dtype=torch.long).to(self.device)
        rewards     = torch.tensor(rewards, dtype=torch.float32).to(self.device)
        next_states = torch.tensor(next_states, dtype=torch.float32).to(self.device)
        dones       = torch.tensor(dones, dtype=torch.float32).to(self.device)

        return states, actions, rewards, next_states, dones, indices


    def update_priorities(self, indices, errors):
        for i, e in zip(indices, errors):
            self.priorities[i] = abs(e) + 1e-5


    def __len__(self):
        return self.size

    def is_ready(self):
        return self.size >= self.batch_size